<a href="https://colab.research.google.com/github/18marimarimari18/DecisionSupportAnalyst-Data-Project-Demo/blob/main/EveryMind_Data_Sentinel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# 1. Create Synthetic Peel Region Intake Data (Ages up to 25)
np.random.seed(42)
n_records = 300

data = {
    'Patient_ID': range(5000, 5000 + n_records),
    'Age': np.random.randint(4, 26, size=n_records), # Adjusted to Peel's age range (up to 25)
    'Symptom_Severity': np.random.uniform(1, 10, size=n_records),
    'Wait_Time_Days': np.random.uniform(5, 60, size=n_records),
    'Referral_Source': np.random.choice(['School', 'Hospital', 'Self-Referral'], size=n_records)
}

df = pd.DataFrame(data)

# 2. Inject "Red Flag" Anomalies (The mistakes your Sentinel will catch)
# Error 1: Out of Age Range (The 85-year-old error)
df.loc[0, 'Age'] = 85

# Error 2: Extreme Wait Time (Data entry typo)
df.loc[1, 'Wait_Time_Days'] = 999

# Insight 1: Urgent Case (High Severity + Almost 0 Wait Time)
# This isn't an "error," it's a priority case that the model should distinguish
df.loc[2, ['Symptom_Severity', 'Wait_Time_Days']] = [10, 0.5]

# 3. Running the ML Sentinel (DBSCAN + Isolation Forest)
features = ['Age', 'Symptom_Severity', 'Wait_Time_Days']
scaler = StandardScaler()
scaled_features = scaler.fit_transform(df[features])

# Method A: DBSCAN (Density-based)
dbscan = DBSCAN(eps=1.2, min_samples=3)
df['DBSCAN_Cluster'] = dbscan.fit_predict(scaled_features)

# Method B: Isolation Forest (Tree-based Outlier detection)
iso_forest = IsolationForest(contamination=0.02, random_state=42)
df['IsoForest_Anomaly'] = iso_forest.fit_predict(scaled_features)

# 4. Final Labeling for Power BI
# We flag it if either model thinks it's an anomaly (-1)
df['Audit_Required'] = np.where((df['DBSCAN_Cluster'] == -1) | (df['IsoForest_Anomaly'] == -1), "Yes", "No")

# Export for your Power BI Web Dashboard
df.to_csv('EveryMind_Sentinel_Data.csv', index=False)
print("Peel Region Sentinel File is ready!")
df.head(10)

Peel Region Sentinel File is ready!


,Patient_ID,Age,Symptom_Severity,Wait_Time_Days,Referral_Source,DBSCAN_Cluster,IsoForest_Anomaly,Audit_Required
0,5000,85,6.680248,52.559731,School,-1,-1,Yes
1,5001,23,8.153302,999.000000,Self-Referral,-1,-1,Yes
2,5002,18,10.000000,0.500000,Self-Referral,0,-1,Yes
3,5003,14,6.192135,59.815026,Self-Referral,0,1,No
4,5004,11,5.432659,35.548744,Self-Referral,0,1,No
5,5005,24,2.757187,47.294308,Hospital,0,1,No
6,5006,10,7.502069,56.962115,School,0,1,No
7,5007,22,3.526951,51.730606,Self-Referral,0,1,No
8,5008,14,1.218844,18.604146,Hospital,0,1,No
9,5009,14,6.809251,29.779927,School,0,1,No
